# 05 — XAI Evaluation — ViT Models (CRC Histology)

Classical XAI evaluation of **ViT-Base/16 · DeiT-Base · DINOv2-ViT-B/14 · Swin-Base**
on the CRC-VAL-HE-7K test set (9 classes).

## Method × Model compatibility

| Method | ViT-Base | DeiT-Base | DINOv2 | Swin-Base |
|---|---|---|---|---|
| Grad-CAM | ✓ | ✓ | ✓ | ✓ |
| Integrated Gradients | ✓ | ✓ | ✓ | ✓ |
| Attention Rollout | ✓ | ✓ | ✓ | ✗ (windowed attn) |
| Generic Attention ⭐ | ✓ | ✓ | ✓ | ✗ |
| LRP / Grad×Input | ✓ | ✓ | ✓ | ✓ |

**Edit only Cell 2** (`USER CONFIG`) to switch models.

**Environment**: Kaggle GPU T4 / P100 — Phase 1 only.

In [ ]:
# 0. Kaggle setup
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru
!pip install -q captum pytorch-grad-cam
!pip install -q PyDrive2
!pip install -q scikit-learn opencv-python-headless

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
# ============================================================
# USER CONFIG — edit this cell only
# ============================================================

# Choose one: vit_base | deit_base | dinov2 | swin_base
MODEL_NAME = "vit_base"

# Google Drive folder containing the checkpoint for this model
DRIVE_FOLDER_ID = "1eq-Jt6O6gO0Ck_oQYwmmc2qrCVLfKlec"  # <-- update as needed

# Dataset subset
N_IMAGES_PER_CLASS = 50
BATCH_SIZE         = 8   # lower than ResNet — ViT uses more VRAM
NUM_WORKERS        = 2
IMAGE_SIZE         = 224
SEED               = 42

# Faithfulness metrics
FAITH_N_STEPS        = 50
INSERTION_BASELINE   = "black"
DELETION_REPLACEMENT = "black"

# Paths (Kaggle)
TRAINVAL_ROOT_STR = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
TEST_ROOT_STR     = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K"
PROJECT_ROOT      = "/kaggle/working/xai-vit-medical"

In [ ]:
# ============================================================
# Per-model XAI configuration
# ============================================================
# All spatial/token parameters are derived from the model configs
# in config/model/*.yaml — do not change unless the model changes.

ALL_METHODS = ["gradcam", "integrated_gradients", "attention_rollout", "generic_attention", "lrp"]
VIT_METHODS = ALL_METHODS                                          # all methods
SWIN_METHODS = ["gradcam", "integrated_gradients", "lrp"]         # no attention methods

MODEL_CONFIGS = {
    "vit_base": {
        "source"            : "timm",
        "timm_name"         : "vit_base_patch16_224",
        "checkpoint_fname"  : "vit_base_best.pth",
        "gradcam_layer_path": "blocks.11.norm1",
        "reshape_h"         : 14,
        "reshape_w"         : 14,
        "num_extra_tokens"  : 1,   # [CLS]
        "methods"           : VIT_METHODS,
    },
    "deit_base": {
        "source"            : "timm",
        "timm_name"         : "deit_base_patch16_224",
        "checkpoint_fname"  : "deit_base_best.pth",
        "gradcam_layer_path": "blocks.11.norm1",
        "reshape_h"         : 14,
        "reshape_w"         : 14,
        "num_extra_tokens"  : 2,   # [CLS] + [DIST]
        "methods"           : VIT_METHODS,
    },
    "dinov2": {
        "source"            : "torch_hub",
        "timm_name"         : None,
        "checkpoint_fname"  : "dinov2_best.pth",
        "gradcam_layer_path": "backbone.blocks.11.norm1",
        "reshape_h"         : 16,  # (224 / 14)^2 = 256 patches → 16×16
        "reshape_w"         : 16,
        "num_extra_tokens"  : 1,   # [CLS]
        "methods"           : VIT_METHODS,
    },
    "swin_base": {
        "source"            : "timm",
        "timm_name"         : "swin_base_patch4_window7_224",
        "checkpoint_fname"  : "swin_base_best.pth",
        "gradcam_layer_path": "layers.3.blocks.1.norm2",
        "reshape_h"         : 7,   # final stage spatial size
        "reshape_w"         : 7,
        "num_extra_tokens"  : 0,   # no CLS token in Swin
        "methods"           : SWIN_METHODS,
    },
}

assert MODEL_NAME in MODEL_CONFIGS, f"Unknown MODEL_NAME: {MODEL_NAME}. Choose from {list(MODEL_CONFIGS)}"
MCFG = MODEL_CONFIGS[MODEL_NAME]
METHODS_TO_RUN = MCFG["methods"]
print(f"Model       : {MODEL_NAME}")
print(f"Source      : {MCFG['source']}")
print(f"Checkpoint  : {MCFG['checkpoint_fname']}")
print(f"GradCAM layer: {MCFG['gradcam_layer_path']}")
print(f"Patch grid  : {MCFG['reshape_h']}×{MCFG['reshape_w']}")
print(f"Extra tokens: {MCFG['num_extra_tokens']}")
print(f"Methods     : {METHODS_TO_RUN}")

In [ ]:
import csv
import gc
import json
import os
import sys
from collections import defaultdict
from functools import partial
from pathlib import Path
from types import SimpleNamespace

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed
from src.xai.classical.gradcam import run_gradcam, vit_reshape_transform
from src.xai.classical.integrated_gradients import run_integrated_gradients
from src.xai.classical.attention_rollout import run_attention_rollout
from src.xai.classical.generic_attention import run_generic_attention
from src.xai.classical.lrp import run_lrp

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES  = len(CLASS_NAMES)

SAVE_DIR = Path(f"{PROJECT_ROOT}/outputs/xai/{MODEL_NAME}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRAINVAL_ROOT = Path(TRAINVAL_ROOT_STR)
TEST_ROOT     = Path(TEST_ROOT_STR)

print(f"Device  : {DEVICE}")
print(f"Classes : {CLASS_NAMES}")
print(f"Output  : {SAVE_DIR}")

In [ ]:
# ── Download checkpoint from Google Drive ──────────────────────────────────
CHECKPOINT_FNAME = MCFG["checkpoint_fname"]
CKPT_LOCAL = Path(f"{PROJECT_ROOT}/outputs/models/{CHECKPOINT_FNAME}")
CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)

if not CKPT_LOCAL.exists():
    file_list = drive.ListFile(
        {"q": f"'{DRIVE_FOLDER_ID}' in parents and title='{CHECKPOINT_FNAME}' and trashed=false"}
    ).GetList()
    if not file_list:
        raise FileNotFoundError(
            f"{CHECKPOINT_FNAME} not found in Drive folder {DRIVE_FOLDER_ID}. "
            "Run the corresponding training notebook first."
        )
    drive_file = file_list[0]
    drive_file.GetContentFile(str(CKPT_LOCAL))
    print(f"Downloaded: {CHECKPOINT_FNAME} → {CKPT_LOCAL}")
else:
    print(f"Checkpoint already present: {CKPT_LOCAL}")

In [ ]:
# ── Build model ────────────────────────────────────────────────────────────

def load_state(model: nn.Module, ckpt_path: Path) -> dict:
    """Load checkpoint weights; return metadata dict."""
    state = torch.load(ckpt_path, map_location="cpu")
    if "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
        return {k: v for k, v in state.items() if k != "model_state_dict"}
    if "state_dict" in state:
        cleaned = {k.replace("module.", "", 1): v for k, v in state["state_dict"].items()}
        model.load_state_dict(cleaned, strict=False)
        return {}
    cleaned = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(cleaned, strict=False)
    return {}


if MCFG["source"] == "torch_hub":
    from src.models.dinov2 import DINOv2Classifier
    model = DINOv2Classifier(num_classes=NUM_CLASSES, usage_mode="frozen_linear_probe")
else:
    model = timm.create_model(MCFG["timm_name"], pretrained=False, num_classes=NUM_CLASSES)

meta = load_state(model, CKPT_LOCAL)
model = model.to(DEVICE).eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model            : {MODEL_NAME}")
print(f"Parameters       : {total_params:,}")
print(f"Checkpoint epoch : {meta.get('epoch', '?')}")
print(f"Checkpoint acc   : {meta.get('val_acc', '?')}")

In [ ]:
# ── Dataset & test subset ──────────────────────────────────────────────────
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=0.25,
    random_state=SEED,
)

test_dataset = CRCHistologyDataset(
    split="test",
    splits=crc_splits,
    image_size=IMAGE_SIZE,
    return_id=True,
)

class_counts: dict[int, int] = defaultdict(int)
subset_indices: list[int] = []
for idx, label in enumerate(test_dataset.labels):
    lbl = int(label)
    if class_counts[lbl] < N_IMAGES_PER_CLASS:
        subset_indices.append(idx)
        class_counts[lbl] += 1
    if all(class_counts[c] >= N_IMAGES_PER_CLASS for c in range(NUM_CLASSES)):
        break

samples_meta: list[tuple[Path, int]] = [test_dataset._samples[i] for i in subset_indices]

eval_dataset = Subset(test_dataset, subset_indices)
eval_loader  = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Images per class:")
for ci, cn in enumerate(CLASS_NAMES):
    print(f"  {cn}: {class_counts[ci]}")
print(f"Total: {len(subset_indices)} | Batches: {len(eval_loader)}")

In [ ]:
# ── XAI configuration objects (replace Hydra for standalone use) ───────────

def _get_module_by_path(root: nn.Module, path: str) -> nn.Module:
    module = root
    for part in path.split("."):
        module = module[int(part)] if part.isdigit() else getattr(module, part)
    return module


def _gradcam_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        variant="GradCAM",
        aug_smooth=False,
        eigen_smooth=False,
        reshape_transform=SimpleNamespace(
            enabled=True,
            height=MCFG["reshape_h"],
            width=MCFG["reshape_w"],
            num_extra_tokens=MCFG["num_extra_tokens"],
        ),
        postprocess=SimpleNamespace(
            relu=True, normalize="minmax",
            upsample_mode="bilinear", upsample_to_input_size=True,
        ),
    )


def _ig_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        n_steps=50,
        internal_batch_size=4,
        method="gausslegendre",
        baseline=SimpleNamespace(type="black_image", blur_sigma=10.0),
        postprocess=SimpleNamespace(
            reduce="sum", abs=True, positive_only=False, normalize="minmax",
        ),
    )


def _rollout_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        head_fusion="mean",
        discard_ratio=0.9,
        residual_weight=0.5,
        num_extra_tokens=MCFG["num_extra_tokens"],
        postprocess=SimpleNamespace(
            reshape_height=MCFG["reshape_h"],
            reshape_width=MCFG["reshape_w"],
            upsample_mode="bilinear",
            upsample_to_input_size=True,
            normalize="minmax",
        ),
    )


def _ga_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        num_extra_tokens=MCFG["num_extra_tokens"],
        start_layer=0,
        postprocess=SimpleNamespace(
            reshape_height=MCFG["reshape_h"],
            reshape_width=MCFG["reshape_w"],
            upsample_mode="bilinear",
            normalize="minmax",
        ),
    )


def _lrp_cfg() -> SimpleNamespace:
    return SimpleNamespace(
        gamma=0.25, epsilon=0.25, alpha=2.0, beta=1.0,
        postprocess=SimpleNamespace(
            reduce="sum", abs=True, positive_only=False, normalize="minmax",
        ),
    )


# Resolve GradCAM target layer once
GRADCAM_TARGET_LAYER = _get_module_by_path(model, MCFG["gradcam_layer_path"])
print(f"GradCAM target layer: {MCFG['gradcam_layer_path']} → {type(GRADCAM_TARGET_LAYER).__name__}")


def compute_saliency(
    method: str,
    images: torch.Tensor,
    targets: list[int],
) -> torch.Tensor:
    """Dispatch to the correct XAI function."""
    if method == "gradcam":
        return run_gradcam(
            model=model,
            images=images,
            targets=targets,
            target_layer=GRADCAM_TARGET_LAYER,
            cfg=_gradcam_cfg(),
        )
    if method == "integrated_gradients":
        return run_integrated_gradients(
            model=model, images=images, targets=targets, cfg=_ig_cfg()
        )
    if method == "attention_rollout":
        return run_attention_rollout(
            model=model, images=images, cfg=_rollout_cfg()
        )
    if method == "generic_attention":
        return run_generic_attention(
            model=model, images=images, targets=targets, cfg=_ga_cfg()
        )
    if method == "lrp":
        return run_lrp(
            model=model, images=images, targets=targets, cfg=_lrp_cfg()
        )
    raise ValueError(f"Unknown method: {method}")


@torch.no_grad()
def model_predict(images: torch.Tensor) -> list[int]:
    out = model(images)
    if hasattr(out, "logits"):
        out = out.logits
    if isinstance(out, tuple):
        out = out[0]
    return torch.argmax(out, dim=1).tolist()


def overlay_heatmap(
    image_chw: torch.Tensor,
    saliency_hw: np.ndarray,
    alpha: float = 0.45,
) -> np.ndarray:
    img = image_chw.detach().cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img  = np.clip(img * std + mean, 0.0, 1.0)
    sal  = saliency_hw.astype(np.float32)
    sal  = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
    heat = cv2.applyColorMap((sal * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


print("XAI helpers ready")

In [ ]:
# ── Faithfulness metrics (Insertion / Deletion AUC) ────────────────────────

def _apply_mask(
    image: torch.Tensor,
    saliency: torch.Tensor,
    fraction: float,
    mode: str,
    baseline: str = "black",
) -> torch.Tensor:
    C, H, W = image.shape
    n_pixels = H * W
    k = max(1, int(fraction * n_pixels))
    flat_sal = saliency.flatten()
    topk_idx = torch.topk(flat_sal, k, largest=True).indices
    mask = torch.zeros(n_pixels, device=image.device)
    mask[topk_idx] = 1.0
    mask = mask.reshape(1, H, W)
    if baseline == "black":
        ref = torch.zeros_like(image)
    elif baseline == "blurred":
        ref = F.avg_pool2d(image.unsqueeze(0), kernel_size=31, stride=1, padding=15).squeeze(0)
    else:
        ref = image.mean(dim=(-2, -1), keepdim=True).expand_as(image)
    return ref * (1 - mask) + image * mask if mode == "insertion" else image * (1 - mask) + ref * mask


@torch.no_grad()
def _class_prob(img_bchw: torch.Tensor, class_idx: int) -> float:
    out = model(img_bchw.to(DEVICE))
    if hasattr(out, "logits"):
        out = out.logits
    if isinstance(out, tuple):
        out = out[0]
    return float(torch.softmax(out, dim=1)[0, class_idx].item())


def insertion_deletion_auc(
    image: torch.Tensor,
    saliency: torch.Tensor,
    target_class: int,
    n_steps: int = FAITH_N_STEPS,
    baseline: str = INSERTION_BASELINE,
) -> dict:
    fractions = np.linspace(0, 1, n_steps + 1)
    ins_probs, del_probs = [], []
    sal_t = saliency.to(image.device)
    for frac in fractions:
        ins_img = _apply_mask(image, sal_t, frac, "insertion", baseline).unsqueeze(0)
        del_img = _apply_mask(image, sal_t, frac, "deletion", baseline).unsqueeze(0)
        ins_probs.append(_class_prob(ins_img, target_class))
        del_probs.append(_class_prob(del_img, target_class))
    ins_auc = float(np.trapz(ins_probs, fractions))
    del_auc = float(np.trapz(del_probs, fractions))
    return {
        "insertion_auc"  : ins_auc,
        "deletion_auc"   : del_auc,
        "faithfulness"   : ins_auc - del_auc,
        "insertion_curve": np.array(ins_probs, dtype=np.float32),
        "deletion_curve" : np.array(del_probs, dtype=np.float32),
    }


print("Faithfulness helpers ready")

In [ ]:
# ── Main evaluation loop ───────────────────────────────────────────────────

def evaluate_method(method_name: str) -> dict:
    method_dir  = SAVE_DIR / method_name
    overlay_dir  = method_dir / "overlays"
    saliency_dir = method_dir / "saliency"
    for d in (method_dir, overlay_dir, saliency_dir):
        d.mkdir(parents=True, exist_ok=True)

    curves_csv = method_dir / "curves.csv"
    auc_csv    = method_dir / "auc_scores.csv"

    with open(curves_csv, "w", newline="") as f:
        csv.writer(f).writerow(["image_id", "method", "step", "type", "probability"])
    with open(auc_csv, "w", newline="") as f:
        csv.writer(f).writerow(
            ["image_id", "label_idx", "label_name", "pred_idx", "pred_name",
             "method", "insertion_auc", "deletion_auc", "faithfulness"]
        )

    ins_aucs, del_aucs = [], []
    ins_curves_list, del_curves_list = [], []
    n_correct   = 0
    n_processed = 0

    for images, labels, image_ids in tqdm(eval_loader, desc=method_name):
        images = images.to(DEVICE, non_blocking=True)
        preds  = model_predict(images)
        targets = preds
        n_correct += sum(p == int(l) for p, l in zip(preds, labels))

        try:
            saliency_batch = compute_saliency(method_name, images, targets)
        except Exception as e:
            raise RuntimeError(f"{method_name} failed on batch: {type(e).__name__}: {e}") from e

        if not torch.is_tensor(saliency_batch):
            saliency_batch = torch.tensor(saliency_batch)
        saliency_batch = saliency_batch.detach().cpu()

        for i in range(images.shape[0]):
            image_id  = image_ids[i]
            label_idx = int(labels[i].item())
            pred_idx  = int(preds[i])
            sal = saliency_batch[i].numpy().astype(np.float32)

            faith = insertion_deletion_auc(
                image=images[i].detach().cpu(),
                saliency=torch.from_numpy(sal),
                target_class=pred_idx,
            )

            ins_aucs.append(faith["insertion_auc"])
            del_aucs.append(faith["deletion_auc"])
            ins_curves_list.append(faith["insertion_curve"])
            del_curves_list.append(faith["deletion_curve"])

            with open(curves_csv, "a", newline="") as f:
                w = csv.writer(f)
                for step, p in enumerate(faith["insertion_curve"]):
                    w.writerow([image_id, method_name, step, "insertion", float(p)])
                for step, p in enumerate(faith["deletion_curve"]):
                    w.writerow([image_id, method_name, step, "deletion", float(p)])

            with open(auc_csv, "a", newline="") as f:
                csv.writer(f).writerow([
                    image_id, label_idx, CLASS_NAMES[label_idx],
                    pred_idx, CLASS_NAMES[pred_idx],
                    method_name,
                    faith["insertion_auc"], faith["deletion_auc"], faith["faithfulness"],
                ])

            stem = Path(image_id).stem
            np.save(saliency_dir / f"{stem}_saliency.npy", sal)
            overlay = overlay_heatmap(images[i].detach().cpu(), sal)
            cv2.imwrite(
                str(overlay_dir / f"{stem}_overlay.jpg"),
                cv2.cvtColor((overlay * 255).astype(np.uint8), cv2.COLOR_RGB2BGR),
            )
            n_processed += 1

    mean_ins = float(np.mean(ins_aucs))
    mean_del = float(np.mean(del_aucs))
    accuracy = n_correct / n_processed
    mean_ins_curve = np.mean(np.stack(ins_curves_list), axis=0)
    mean_del_curve = np.mean(np.stack(del_curves_list), axis=0)

    summary = {
        "model"              : MODEL_NAME,
        "method"             : method_name,
        "n_images"           : n_processed,
        "accuracy"           : accuracy,
        "mean_insertion_auc" : mean_ins,
        "mean_deletion_auc"  : mean_del,
        "mean_faithfulness"  : mean_ins - mean_del,
    }
    with open(method_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    x = np.linspace(0, 1, len(mean_ins_curve))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(x, mean_ins_curve)
    axes[0].set(title=f"{MODEL_NAME} / {method_name} — Mean Insertion Curve",
                xlabel="Fraction of pixels inserted", ylabel="Class probability")
    axes[0].grid(True)
    axes[1].plot(x, mean_del_curve, color="tab:red")
    axes[1].set(title=f"{MODEL_NAME} / {method_name} — Mean Deletion Curve",
                xlabel="Fraction of pixels deleted", ylabel="Class probability")
    axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(str(method_dir / "curves.png"), dpi=150)
    plt.show()

    print(f"[{method_name}] n={n_processed}  acc={accuracy:.3f}")
    print(f"[{method_name}] insertion AUC = {mean_ins:.4f}")
    print(f"[{method_name}] deletion  AUC = {mean_del:.4f}")
    print(f"[{method_name}] faithfulness  = {mean_ins - mean_del:.4f}")
    return summary


print("evaluate_method() ready")

In [ ]:
# ── Run all applicable methods ─────────────────────────────────────────────
all_summaries: list[dict] = []

for method in METHODS_TO_RUN:
    print("\n" + "=" * 72)
    print(f"Running: {method}")
    print("=" * 72)
    try:
        summary = evaluate_method(method)
        all_summaries.append(summary)
    except Exception as e:
        print(f"[ERROR] {method}: {type(e).__name__}: {e}")
        all_summaries.append({"model": MODEL_NAME, "method": method, "error": str(e)})

skipped_methods = [m for m in ALL_METHODS if m not in METHODS_TO_RUN]
for m in skipped_methods:
    all_summaries.append({"model": MODEL_NAME, "method": m, "skipped": True,
                          "reason": "Not applicable to this model family"})

print("\n" + "=" * 72)
print(f"Summary — {MODEL_NAME}")
print("=" * 72)
for s in all_summaries:
    if s.get("skipped"):
        print(f"  {s['method']:25s} SKIPPED ({s['reason']})")
    elif "error" in s:
        print(f"  {s['method']:25s} ERROR: {s['error']}")
    else:
        print(
            f"  {s['method']:25s} "
            f"ins={s['mean_insertion_auc']:.4f}  "
            f"del={s['mean_deletion_auc']:.4f}  "
            f"faith={s['mean_faithfulness']:.4f}"
        )

In [ ]:
# ── Visual inspection — sample overlays ───────────────────────────────────
N_DISPLAY = 3

for s in all_summaries:
    if "error" in s or s.get("skipped"):
        continue
    method = s["method"]
    overlay_dir = SAVE_DIR / method / "overlays"
    candidates  = sorted(overlay_dir.glob("*.jpg"))[:N_DISPLAY]
    if not candidates:
        continue
    fig, axes = plt.subplots(1, len(candidates), figsize=(5 * len(candidates), 5))
    if len(candidates) == 1:
        axes = [axes]
    fig.suptitle(f"{MODEL_NAME} — {method}", fontsize=13)
    for ax, p in zip(axes, candidates):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(p.stem.replace("_overlay", ""), fontsize=7)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Per-class saliency montage ─────────────────────────────────────────────

def show_class_montage(method_name: str, n_per_class: int = 1) -> None:
    overlay_dir = SAVE_DIR / method_name / "overlays"
    if not overlay_dir.exists():
        print(f"No overlays for {method_name}")
        return
    class_to_paths: dict[int, list[Path]] = defaultdict(list)
    for img_path, label_idx in samples_meta:
        overlay = overlay_dir / f"{img_path.stem}_overlay.jpg"
        if overlay.exists():
            class_to_paths[label_idx].append(overlay)

    fig, axes = plt.subplots(
        NUM_CLASSES, n_per_class,
        figsize=(4 * n_per_class, 4 * NUM_CLASSES),
        squeeze=False,
    )
    fig.suptitle(f"{MODEL_NAME} — {method_name} — per-class overlays", fontsize=13)
    for row, class_idx in enumerate(range(NUM_CLASSES)):
        paths = class_to_paths[class_idx][:n_per_class]
        for col in range(n_per_class):
            ax = axes[row][col]
            if col < len(paths):
                ax.imshow(cv2.cvtColor(cv2.imread(str(paths[col])), cv2.COLOR_BGR2RGB))
                if col == 0:
                    ax.set_ylabel(CLASS_NAMES[class_idx], fontsize=10,
                                  rotation=0, labelpad=55, va="center")
            else:
                ax.set_visible(False)
            ax.axis("off")
    plt.tight_layout()
    out = SAVE_DIR / method_name / "class_montage.png"
    plt.savefig(str(out), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")


for s in all_summaries:
    if "error" not in s and not s.get("skipped"):
        show_class_montage(s["method"], n_per_class=2)

In [ ]:
# ── Save global results ─────────────────────────────────────────────────────
global_summary_path = SAVE_DIR / "summary_all_methods.json"
with open(global_summary_path, "w") as f:
    json.dump(all_summaries, f, indent=2)

manifest_path = SAVE_DIR / "subset_manifest.csv"
with open(manifest_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image_path", "label_idx", "label_name"])
    for img_path, label_idx in samples_meta:
        w.writerow([str(img_path), label_idx, CLASS_NAMES[label_idx]])

print(f"Saved: {global_summary_path}")
print(f"Saved: {manifest_path}")

In [ ]:
# ── Upload results to Google Drive ─────────────────────────────────────────
RESULTS_DRIVE_FOLDER = DRIVE_FOLDER_ID  # same folder as checkpoint; update if needed

files_to_upload = [global_summary_path, manifest_path]
for s in all_summaries:
    if "error" in s or s.get("skipped"):
        continue
    method_dir = SAVE_DIR / s["method"]
    files_to_upload += [
        method_dir / "summary.json",
        method_dir / "curves.png",
        method_dir / "auc_scores.csv",
        method_dir / "class_montage.png",
    ]

for fpath in files_to_upload:
    fpath = Path(fpath)
    if not fpath.exists():
        print(f"  Not found (skipped): {fpath.name}")
        continue
    drive_file = drive.CreateFile({
        "title"  : f"{MODEL_NAME}_{fpath.name}",
        "parents": [{"id": RESULTS_DRIVE_FOLDER}],
    })
    drive_file.SetContentFile(str(fpath))
    drive_file.Upload()
    print(f"  Uploaded: {MODEL_NAME}_{fpath.name}")

print("Done.")

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup done.")